# Chapter 5 — Backpropagation

**The building block this notebook untangles:** Chapter 4 gave us a loss — one
number saying how wrong the network is. To improve, we need to know, for
*every single weight*, "if I nudge this weight up slightly, does the loss go up
or down, and by how much?" That sensitivity is the weight's **gradient**.
Backpropagation is just the **chain rule from calculus**, applied systematically
from the output of the network back to its first layer, so every weight's
gradient gets computed in one efficient backward sweep.

## Part 1 — The chain rule, on one scalar path

Before tackling the whole network, do it on the smallest possible example: one
input `x`, one weight `w`, one bias `b`, a sigmoid, and an MSE loss against one
target `y`.

$$z = wx + b \qquad a = \sigma(z) \qquad L = (a - y)^2$$

The chain rule says the effect of `w` on `L` is the *product* of each step's
local effect:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x, w, b, y = 2.0, 0.5, -0.1, 1.0

# --- forward pass, keeping every intermediate value ---
z = w * x + b
a = sigmoid(z)
L = (a - y) ** 2

# --- backward pass: chain rule, one factor per step, walking backward ---
dL_da = 2 * (a - y)          # d/da (a - y)^2
da_dz = a * (1 - a)          # sigmoid derivative, in terms of its own output
dz_dw = x                    # d/dw (w*x + b)

dL_dw = dL_da * da_dz * dz_dw

print(f"forward:  z={z:.4f}  a={a:.4f}  L={L:.4f}")
print(f"backward: dL/da={dL_da:.4f}  da/dz={da_dz:.4f}  dz/dw={dz_dw:.4f}")
print(f"chain rule gradient  dL/dw = {dL_dw:.6f}")

forward:  z=0.9000  a=0.7109  L=0.0836
backward: dL/da=-0.5781  da/dz=0.2055  dz/dw=2.0000
chain rule gradient  dL/dw = -0.237600


## Part 2 — Trust, but verify: numerical gradient checking

Anyone can *write* a derivative formula; the classic sanity check is to also
compute the gradient the slow, dumb way — nudge `w` by a tiny amount and measure
how much `L` actually moved — and confirm the two agree. This is the standard
way to catch backprop bugs, and we'll reuse it below for the full network.

$$\frac{\partial L}{\partial w} \approx \frac{L(w+\epsilon) - L(w-\epsilon)}{2\epsilon}$$

In [2]:
def loss_at(w_val):
    z = w_val * x + b
    a = sigmoid(z)
    return (a - y) ** 2

eps = 1e-5
numerical_grad = (loss_at(w + eps) - loss_at(w - eps)) / (2 * eps)

print(f"analytic (chain rule) gradient: {dL_dw:.6f}")
print(f"numerical (finite diff) gradient: {numerical_grad:.6f}")
print(f"difference: {abs(dL_dw - numerical_grad):.2e}")

analytic (chain rule) gradient: -0.237600
numerical (finite diff) gradient: -0.237600
difference: 1.51e-11


Matches to 6+ decimal places. That agreement is the whole trick of
backpropagation validated on the smallest possible example — now we scale the
same idea up to a full layer.

## Part 3 — Backprop through the 2-layer network

Same network as Chapters 3–4: `2 inputs → 4 hidden (ReLU) → 1 output (sigmoid)`,
trained with binary cross-entropy loss. We derive the backward pass one layer at
a time, walking from the loss back to `W1`:

- `dL/dZ2 = A2 - y` (the BCE + sigmoid combination famously simplifies to this)
- `dL/dW2 = A1ᵀ · dL/dZ2`,  `dL/db2 = sum(dL/dZ2)`
- `dL/dA1 = dL/dZ2 · W2ᵀ` — the error flowing backward *into* the hidden layer
- `dL/dZ1 = dL/dA1 * relu′(Z1)` — gated by which neurons were even active
- `dL/dW1 = Xᵀ · dL/dZ1`,  `dL/db1 = sum(dL/dZ1)`

Each line is the chain rule, one layer at a time — exactly what we did by hand
in Part 1, just with matrices instead of scalars.

In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    return Z1, A1, Z2, A2

def bce_loss(A2, y):
    y = y.reshape(A2.shape)
    eps = 1e-12
    p = np.clip(A2, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def backward(X, y, Z1, A1, Z2, A2, W2):
    n = X.shape[0]
    y = y.reshape(A2.shape)

    dZ2 = (A2 - y) / n                    # (n, 1)
    dW2 = A1.T @ dZ2                      # (hidden, 1)
    db2 = dZ2.sum(axis=0)                 # (1,)

    dA1 = dZ2 @ W2.T                      # (n, hidden)
    dZ1 = dA1 * relu_deriv(Z1)            # (n, hidden)
    dW1 = X.T @ dZ1                       # (in, hidden)
    db1 = dZ1.sum(axis=0)                 # (hidden,)

    return dW1, db1, dW2, db2

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

rng = np.random.default_rng(7)
W1 = rng.normal(scale=1.0, size=(2, 4))
b1 = np.zeros(4)
W2 = rng.normal(scale=1.0, size=(4, 1))
b2 = np.zeros(1)

Z1, A1, Z2, A2 = forward(X_xor, W1, b1, W2, b2)
dW1, db1, dW2, db2 = backward(X_xor, y_xor, Z1, A1, Z2, A2, W2)

print("dW1 shape:", dW1.shape, " dW2 shape:", dW2.shape)
print("\ndL/dW1 =\n", np.round(dW1, 4))

dW1 shape: (2, 4)  dW2 shape: (4, 1)

dL/dW1 =
 [[ 0.0672  0.0848  0.      0.0482]
 [ 0.      0.     -0.046   0.0147]]


## Part 4 — Gradient-checking the whole network

Same trust-but-verify move as Part 2, now applied to every entry of `W1`: nudge
each weight individually, measure the loss change, and compare against the
analytic gradient from `backward()`.

In [4]:
def numerical_gradient_W1(X, y, W1, b1, W2, b2, eps=1e-5):
    grad = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W1_plus = W1.copy();  W1_plus[i, j] += eps
            W1_minus = W1.copy(); W1_minus[i, j] -= eps

            _, _, _, A2_plus  = forward(X, W1_plus,  b1, W2, b2)
            _, _, _, A2_minus = forward(X, W1_minus, b1, W2, b2)

            loss_plus  = bce_loss(A2_plus, y)
            loss_minus = bce_loss(A2_minus, y)
            grad[i, j] = (loss_plus - loss_minus) / (2 * eps)
    return grad

numerical_dW1 = numerical_gradient_W1(X_xor, y_xor, W1, b1, W2, b2)

print("analytic dW1:\n", np.round(dW1, 6))
print("\nnumerical dW1:\n", np.round(numerical_dW1, 6))
print("\nmax absolute difference:", np.max(np.abs(dW1 - numerical_dW1)))

analytic dW1:
 [[ 0.06723   0.08475   0.        0.048182]
 [ 0.        0.       -0.046011  0.01466 ]]

numerical dW1:
 [[ 0.06723   0.08475   0.        0.048182]
 [ 0.        0.       -0.046011  0.01466 ]]

max absolute difference: 5.6263761011310365e-12


The max difference should be tiny (well under `1e-6`) — proof that the
matrix-form backward pass really does compute the same gradients the chain
rule guarantees, just efficiently.

## Recap

Backpropagation is the chain rule applied layer by layer, backward from the
loss. We now have, for any set of weights, both a forward pass (Chapter 3) and
the exact gradient of the loss with respect to every weight. Chapter 6 uses that
gradient to actually *update* the weights — the last piece of the training loop.

**Next:** [Chapter 6 — Gradient Descent & the Training Loop](06_gradient_descent_training_loop.ipynb)